In [0]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T

# Choose which Delta mapping table this run may update. The two service_group_* tables
# are test copies for the 'new ID' and 'changed mapping' scenarios. The default is
# the production service_groups table so the existing scheduled job keeps its target.
# Select a test table in the widget before running all cells to exercise that case.
dbutils.widgets.dropdown(
    "service_group_table",
    "service_groups",
    ["service_groups", "service_group_newid", "service_group_change"],
    "Mapping table",
)
# The allowlist prevents an unexpected widget value from becoming a table name.
# All three tables are assumed to have the same six mapping columns and to live in
# opsanalytics_adb_workspace01.capacity_modeling.
target_name = dbutils.widgets.get("service_group_table")
if target_name not in {"service_groups", "service_group_newid", "service_group_change"}:
    raise ValueError(f"Unsupported service group table: {target_name}")
TARGET = f"opsanalytics_adb_workspace01.capacity_modeling.{target_name}"
MAPPING_COLUMNS = [
    "LOC_NAME", "EPIC_DEPT_ID", "EXTERNAL_NAME", "SERVICE_GROUP", "VALID_FROM", "VALID_TO"
]
# Use one Spark run date for every VALID_FROM / VALID_TO written in this run.
# Remember the target's Delta version: if another run changes the table after
# this notebook reads it, the final cell stops instead of applying stale results.
run_date = spark.range(1).select(F.current_date().alias("run_date")).first().run_date
starting_version = DeltaTable.forName(spark, TARGET).history(1).select("version").first().version


In [0]:
# Read the complete mapping history and the current Oracle department reference.
# Read every row, including expired rows. We compare only active rows below, but
# Delta MERGE keeps the expired history already present in the target table.
current = spark.table(TARGET).select(*MAPPING_COLUMNS)

# Read the live Oracle reference each run. Secrets stay in Databricks; the
# notebook contains only secret names, never the URL or password values.
oracle_jdbc_url = dbutils.secrets.get(scope="oao_secrets", key="ORACLE_JDBC_URL")
oracle_password = dbutils.secrets.get(scope="oao_secrets", key="OAO_PRODUCTION")

# Shared JDBC reader for the department reference and the aggregate charge
# lookup. The charge lookup selects department IDs only, not patient-level rows.
def oracle_table(dbtable):
    return (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", dbtable)
        .option("user", "OAO_PRODUCTION")
        .option("password", oracle_password)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("oracle.net.ssl_server_dn_match", "true")
        .option("fetchsize", "10000")
        .load()
    )

# A trailing space or an empty string should not look like a changed mapping.
# Convert IDs to strings so Oracle and Delta can be joined even if their numeric
# column types differ. Empty values become null for consistent comparison.
def normalized(column):
    value = F.trim(F.col(column).cast("string"))
    return F.when(F.length(value) == 0, F.lit(None)).otherwise(value)

# These four Oracle fields define the latest desired mapping. DEPARTMENT_ID
# corresponds to EPIC_DEPT_ID in the Delta target. Other reference columns are
# intentionally ignored because the mapping table does not store them.
clarity_dep_ref = (
    oracle_table("(SELECT DEPARTMENT_ID, LOC_NAME, EXTERNAL_NAME, SERVICE_GROUP "
                 "FROM DASHBD_USER.CLARITY_DEP_REF) dept_ref")
    .select(
        normalized("DEPARTMENT_ID").alias("EPIC_DEPT_ID"),
        normalized("LOC_NAME").alias("LOC_NAME"),
        normalized("EXTERNAL_NAME").alias("EXTERNAL_NAME"),
        normalized("SERVICE_GROUP").alias("SERVICE_GROUP"),
    )
    .cache()
)

# A missing/duplicated reference ID would make the update ambiguous. Stop
# before any write so the job cannot silently choose one of two mappings.
if clarity_dep_ref.count() == 0:
    raise ValueError("Clarity department reference is empty")
if clarity_dep_ref.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Clarity department reference has duplicate department IDs")

# VALID_TO = null marks the current mapping for a department. Historical
# rows with a VALID_TO date remain in the table and do not take part in the join.
active = current.filter(F.col("VALID_TO").isNull()).withColumn(
    "EPIC_DEPT_ID", normalized("EPIC_DEPT_ID")
).cache()
if active.filter(F.col("EPIC_DEPT_ID").isNull()).limit(1).count():
    raise ValueError("Active service group mapping has a blank department ID")
if active.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Active service group mapping has duplicate department IDs")


In [0]:
# Compare active, non-emergency mappings. Missing reference rows do not retire mappings.
# Intent check: preserve the legacy rule that existing departments whose
# current EXTERNAL_NAME contains EMERGENCY are not updated by this process.
# They still count as existing IDs, so they will not be mistaken for new ones.
tracked = active.filter(
    ~F.upper(F.coalesce(F.col("EXTERNAL_NAME"), F.lit(""))).contains("EMERGENCY")
)
# Give current and Oracle values distinct names so the comparison is explicit.
# VALID_FROM is retained to decide later whether to close history or replace a
# row that was already created today.
old = tracked.select(
    "EPIC_DEPT_ID",
    normalized("LOC_NAME").alias("old_loc"),
    normalized("EXTERNAL_NAME").alias("old_name"),
    normalized("SERVICE_GROUP").alias("old_group"),
    F.col("VALID_FROM").cast("date").alias("old_valid_from"),
)
ref = clarity_dep_ref.select(
    "EPIC_DEPT_ID",
    F.col("LOC_NAME").alias("new_loc"),
    F.col("EXTERNAL_NAME").alias("new_name"),
    F.col("SERVICE_GROUP").alias("new_group"),
)
# Inner join = IDs present in both active mappings and Oracle. Compare only
# location, department name, and service group, after trimming whitespace.
# Null-safe equality treats two missing values as equal.
changes = (
    old.join(ref, "EPIC_DEPT_ID", "inner")
    .filter(
        ~F.col("old_loc").eqNullSafe(F.col("new_loc"))
        | ~F.col("old_name").eqNullSafe(F.col("new_name"))
        | ~F.col("old_group").eqNullSafe(F.col("new_group"))
    )
    .cache()
)

# Closing a row with an unknown or future start date would corrupt its
# validity interval, so fail before preparing any write.
if changes.filter(F.col("old_valid_from").isNull() | (F.col("old_valid_from") > F.lit(run_date))).limit(1).count():
    raise ValueError("A changed mapping has a missing or future VALID_FROM date")

# Show the proposed old/new values before the write cell. These are
# department-level mapping fields only, not charge-detail records.
print(f"Changed mappings: {changes.count()}; new reference departments: {new_candidates.count()}")
display(changes.select("EPIC_DEPT_ID", "old_loc", "new_loc", "old_name", "new_name", "old_group", "new_group"))


In [0]:
# Left anti join = reference IDs with no active target mapping. A department
# with only expired history is eligible to be re-added. An active ID missing
# from Oracle is deliberately left unchanged; this process does not retire it.
# Intent check: a new EMERGENCY department can still qualify through bed charges.
new_candidates = clarity_dep_ref.join(active.select("EPIC_DEPT_ID"), "EPIC_DEPT_ID", "left_anti").cache()

# Check only new department IDs in authoritative Oracle charges. Return IDs, never charge rows.
# The start date follows the legacy service-group process.
# Keep this query narrow: ask Oracle about only the candidate IDs, instead
# of scanning or collecting the full OE_CHARGE_DETAIL table in Databricks.
# The stored Databricks charge copy is filtered and may not be current enough
# for deciding whether a new department has ever had a bed charge.
candidate_ids = [row.EPIC_DEPT_ID for row in new_candidates.select("EPIC_DEPT_ID").collect()]
if any(not department_id.isascii() or not department_id.isdigit() for department_id in candidate_ids):
    raise ValueError("A new department ID is not numeric; cannot safely construct the Oracle filter")

# The fixed 2024-06-01 threshold and BED CHARGES category come from the Rmd.
# With no candidate IDs, skip Oracle entirely (an empty IN list is invalid SQL).
bed_department_ids = set()
if candidate_ids:
    quoted_ids = ", ".join("'" + department_id + "'" for department_id in candidate_ids)
    query = (
        "(SELECT DISTINCT TO_CHAR(EPIC_DEPT_ID) AS EPIC_DEPT_ID "
        "FROM OE_CHARGE_DETAIL "
        "WHERE BILLING_CAT_DESC = 'BED CHARGES' "
        "AND SERVICE_DATE >= 20240601 "
        f"AND EPIC_DEPT_ID IN ({quoted_ids})) bed_depts"
    )
    bed_department_ids.update(
        row.EPIC_DEPT_ID.strip()
        for row in oracle_table(query).select("EPIC_DEPT_ID").collect()
        if row.EPIC_DEPT_ID is not None
    )

# Retain only reference rows whose IDs Oracle returned. A new department
# without a qualifying charge is shown as a candidate above but is not written.
# An explicit schema also lets the empty-result case work without special code.
bed_id_schema = T.StructType([T.StructField("EPIC_DEPT_ID", T.StringType(), False)])
bed_ids = spark.createDataFrame([(department_id,) for department_id in sorted(bed_department_ids)], bed_id_schema)
new_with_bed_charges = new_candidates.join(F.broadcast(bed_ids), "EPIC_DEPT_ID", "left_semi").cache()
print(f"New departments with bed charges: {new_with_bed_charges.count()}")
display(new_with_bed_charges)


In [0]:
# Validate the rows that would be written. Keep all historical rows in the target.
# Build the exact mappings we would insert/update, using the Oracle values
# for changed rows and the reference values for newly qualified departments.
# No target writes have happened yet.
changed_rows = changes.select(
    "EPIC_DEPT_ID",
    F.col("new_loc").alias("LOC_NAME"),
    F.col("new_name").alias("EXTERNAL_NAME"),
    F.col("new_group").alias("SERVICE_GROUP"),
    "old_valid_from",
)
new_rows = new_with_bed_charges.select(
    "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP",
    F.lit(None).cast("date").alias("old_valid_from"),
)
# Intent check: a missing service group (or name/location) blocks the whole
# update rather than creating a partially classified active mapping.
to_write = changed_rows.unionByName(new_rows).cache()
if to_write.filter(
    F.col("LOC_NAME").isNull()
    | F.col("EXTERNAL_NAME").isNull()
    | F.col("SERVICE_GROUP").isNull()
).limit(1).count():
    raise ValueError("A mapping to write is missing a location, department name, or service group")

# Date-granularity history cannot close a row before its start. Update same-day
# changes in place; close older rows yesterday and insert today's replacement.
# A change to a row started before today gets two actions: expire that
# active row yesterday, then insert the new mapping starting today. A second
# change on the same day updates today's row in place, because yesterday would
# be earlier than its VALID_FROM date.
older_changes = changed_rows.filter(F.col("old_valid_from") < F.lit(run_date))
same_day_changes = changed_rows.filter(F.col("old_valid_from") == F.lit(run_date))

# Stage 'close' and 'replace' rows with a merge key so they match the active
# target row. Stage 'insert' rows with a null merge key so MERGE treats them as
# new rows, even when they share an ID with a row being expired in this run.
def stage(frame, operation, merge_key):
    return frame.select(
        F.lit(operation).alias("operation"),
        (F.col("EPIC_DEPT_ID") if merge_key else F.lit(None).cast("string")).alias("merge_key"),
        "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP",
    )

staged = (
    stage(older_changes, "close", True)
    .unionByName(stage(same_day_changes, "replace", True))
    .unionByName(stage(older_changes, "insert", False))
    .unionByName(stage(new_rows, "insert", False))
)

# No differences means no Delta write or new table version. On a write,
# recheck the version captured in cell 1; rerun all cells if the table changed.
# The MERGE applies history closure and insertion as one Delta transaction.
if to_write.limit(1).count():
    target = DeltaTable.forName(spark, TARGET)
    latest_version = target.history(1).select("version").first().version
    if latest_version != starting_version:
        raise RuntimeError("Service group table changed since this notebook read it; rerun all cells")
    (
        target.alias("t")
        .merge(
            staged.alias("s"),
            "CAST(t.EPIC_DEPT_ID AS STRING) = s.merge_key AND t.VALID_TO IS NULL",
        )
        .whenMatchedUpdate(
            condition="s.operation = 'close'",
            set={"VALID_TO": F.lit(run_date - timedelta(days=1))},
        )
        .whenMatchedUpdate(
            condition="s.operation = 'replace'",
            set={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
            },
        )
        # New rows include today's VALID_FROM and an open-ended VALID_TO.
        # Unchanged departments and expired historical rows are untouched.
        .whenNotMatchedInsert(
            condition="s.operation = 'insert'",
            values={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EPIC_DEPT_ID": F.col("s.EPIC_DEPT_ID"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
                "VALID_FROM": F.lit(run_date),
                "VALID_TO": F.lit(None).cast(current.schema["VALID_TO"].dataType),
            },
        )
        .execute()
    )
    print("Service group mappings updated")
else:
    print("No service group mappings to update")
